# Cattle breed classification on Colab

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Then run the cells from top to bottom.

- The code comes from GitHub, so push your latest work from your PC first.
- Datasets and trained models are kept in **Google Drive → My Drive → cattle-breed**. They survive disconnects and are only downloaded once.
- After a disconnect, run the setup cell again and continue where you stopped. Training resumes on its own.

## Setup (run every time)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

%cd /content
!test -d CattleClass || git clone -q https://github.com/ShayanAhmed12/CattleClass.git
%cd /content/CattleClass
!git pull -q
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name --format=csv,noheader || echo "No GPU! Runtime > Change runtime type > T4 GPU"
!python -m pytest -q

## Step 1: data
The first time this downloads the three datasets into your Drive (about 10 minutes). After that it only checks them.

In [ ]:
!python prepare_data.py

## Step 2: training (B0, E1, E2, final model)
If Colab disconnects, run Setup again and then this cell. Folds that are already finished are skipped.

In [ ]:
!if [ -f train.py ]; then python train.py; else echo "train.py is not written yet (T7 in docs/PLAN.md)"; fi

In [ ]:
!if [ -f train.py ]; then python train.py --final; else echo "train.py is not written yet (T8 in docs/PLAN.md)"; fi

## Step 3: evaluation (E3, E4, E5, figures)

In [ ]:
!if [ -f evaluate.py ]; then python evaluate.py; else echo "evaluate.py is not written yet (T9 in docs/PLAN.md)"; fi

## Step 4: demo
Prints a public link you can open on any device.

In [ ]:
!if [ -f demo.py ]; then python demo.py --share; else echo "demo.py is not written yet (T12 in docs/PLAN.md)"; fi

## Save the results

`results/` (metrics and figures) lives in this temporary Colab copy of the repository, so save it before you close the tab.

**Option A, simple:** copy it to Drive. Then, on your PC, download *My Drive/cattle-breed/results* into the project's `results/` folder.

In [ ]:
!mkdir -p /content/drive/MyDrive/cattle-breed/results && cp -r results/. /content/drive/MyDrive/cattle-breed/results/ && echo saved

**Option B:** push it to GitHub, then `git pull` on your PC.

This needs a one-time setup:
1. On GitHub, open *Settings → Developer settings → Fine-grained tokens* and create a token with *Contents: read and write* on CattleClass.
2. In Colab, open the key icon on the left and add the token as a secret named `GH_TOKEN`.

In [ ]:
from google.colab import userdata
import subprocess
token = userdata.get("GH_TOKEN")
!git config user.name "ShayanAhmed12"
!git config user.email "shayanahmed12999@gmail.com"
!git add results && git commit -q -m "Results from Colab"
r = subprocess.run(["git", "push", "-q", f"https://{token}@github.com/ShayanAhmed12/CattleClass.git", "HEAD:main"],
                   capture_output=True, text=True)
print("pushed" if r.returncode == 0 else "push failed: run  git pull  on Colab first, or use Option A")